In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('incidence-rate-data.csv')

In [3]:
df1 = pd.read_csv('coverage-data.csv',encoding='latin1')  
df2 = pd.read_csv('vaccine-introduction-data.csv',encoding='latin1')
df3 = pd.read_csv('vaccine-schedule-data.csv',encoding='latin1')
df4 = pd.read_csv('reported-cases-data.csv',encoding='latin1')

In [4]:
df1.isnull().sum()

GROUP                                 0
CODE                                  1
NAME                               1275
YEAR                                  1
ANTIGEN                               1
ANTIGEN_DESCRIPTION                   1
COVERAGE_CATEGORY                     1
COVERAGE_CATEGORY_DESCRIPTION         1
TARGET_NUMBER                    320829
DOSES                            320532
COVERAGE                         169382
dtype: int64

In [5]:
df1.head(20)  

,GROUP,CODE,NAME,YEAR,ANTIGEN,ANTIGEN_DESCRIPTION,COVERAGE_CATEGORY,COVERAGE_CATEGORY_DESCRIPTION,TARGET_NUMBER,DOSES,COVERAGE
0,COUNTRIES,ABW,Aruba,2023.0,BCG,BCG,ADMIN,Administrative coverage,NaN,NaN,NaN
1,COUNTRIES,ABW,Aruba,2023.0,BCG,BCG,OFFICIAL,Official coverage,NaN,NaN,NaN
2,COUNTRIES,ABW,Aruba,2023.0,DIPHCV4,"Diphtheria-containing vaccine, 4th dose (1st b...",ADMIN,Administrative coverage,1044.0,945.0,90.52
3,COUNTRIES,ABW,Aruba,2023.0,DIPHCV4,"Diphtheria-containing vaccine, 4th dose (1st b...",OFFICIAL,Official coverage,NaN,NaN,90.52
4,COUNTRIES,ABW,Aruba,2023.0,DIPHCV5,"Diphtheria-containing vaccine, 5th dose (2nd b...",ADMIN,Administrative coverage,1219.0,1008.0,82.69
5,COUNTRIES,ABW,Aruba,2023.0,DIPHCV5,"Diphtheria-containing vaccine, 5th dose (2nd b...",OFFICIAL,Official coverage,NaN,NaN,82.69
6,COUNTRIES,ABW,Aruba,2023.0,DIPHCV6,"Diphtheria-containing vaccine, 6th dose (3rd b...",ADMIN,Administrative coverage,1468.0,1219.0,83.04
7,COUNTRIES,ABW,Aruba,2023.0,DIPHCV6,"Diphtheria-containing vaccine, 6th dose (3rd b...",OFFICIAL,Official coverage,NaN,NaN,83.04
8,COUNTRIES,ABW,Aruba,2023.0,DTPCV1,"DTP-containing vaccine, 1st dose",ADMIN,Administrative coverage,945.0,926.0,97.99
9,COUNTRIES,ABW,Aruba,2023.0,DTPCV1,"DTP-containing vaccine, 1st dose",OFFICIAL,Official coverage,NaN,NaN,97.99


In [6]:
import pandas as pd
import numpy as np
from datetime import datetime

print("="*80)
print("VACCINATION DATA CLEANING SCRIPT")
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)


print("\nSTEP 1: Loading data files...")

coverage_data = pd.read_csv('coverage-data.csv',encoding='latin1')
incidence_rate = pd.read_csv('incidence-rate-data.csv',encoding='latin1')
reported_cases = pd.read_csv('reported-cases-data.csv',encoding='latin1')
vaccine_intro = pd.read_csv('vaccine-introduction-data.csv',encoding='latin1')
vaccine_schedule = pd.read_csv('vaccine-schedule-data.csv',encoding='latin1')

print(f"✓ Coverage Data: {len(coverage_data):,} rows loaded")
print(f"✓ Incidence Rate: {len(incidence_rate):,} rows loaded")
print(f"✓ Reported Cases: {len(reported_cases):,} rows loaded")
print(f"✓ Vaccine Introduction: {len(vaccine_intro):,} rows loaded")
print(f"✓ Vaccine Schedule: {len(vaccine_schedule):,} rows loaded")

original_counts = {
    'coverage': len(coverage_data),
    'incidence': len(incidence_rate),
    'cases': len(reported_cases),
    'intro': len(vaccine_intro),
    'schedule': len(vaccine_schedule)
}


print("\n" + "="*80)
print("STEP 2: CLEANING COVERAGE DATA")
print("="*80)


print("\n2.1: Removing rows with NULL in key columns...")
before = len(coverage_data)
coverage_data = coverage_data.dropna(subset=['CODE', 'YEAR', 'ANTIGEN', 'COVERAGE_CATEGORY'])
after = len(coverage_data)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

print("\n2.2: Fixing negative DOSES values...")
negative_mask = coverage_data['DOSES'] < 0
negative_count = negative_mask.sum()
if negative_count > 0:
    print(f"  Found {negative_count} negative values")
    print(f"  Range: {coverage_data.loc[negative_mask, 'DOSES'].min():,.0f} to {coverage_data.loc[negative_mask, 'DOSES'].max():,.0f}")
    coverage_data.loc[negative_mask, 'DOSES'] = np.nan
    print(f"  ✓ Converted to NULL")
else:
    print(f"  ✓ No negative values found")

print("\n2.3: Fixing impossible COVERAGE percentages...")

severe_mask = coverage_data['COVERAGE'] > 150
moderate_mask = (coverage_data['COVERAGE'] > 100) & (coverage_data['COVERAGE'] <= 150)

severe_count = severe_mask.sum()
moderate_count = moderate_mask.sum()

print(f"  Severe errors (>150%): {severe_count:,} rows")
print(f"  Moderate (100-150%): {moderate_count:,} rows")

if severe_count > 0:
    coverage_data.loc[severe_mask, 'COVERAGE'] = np.nan
    print(f"  ✓ Set >150% to NULL")

if moderate_count > 0:
    coverage_data.loc[moderate_mask, 'COVERAGE'] = 100.0
    print(f"  ✓ Capped 100-150% values at 100%")

print("\n2.4: Standardizing text columns...")
coverage_data['CODE'] = coverage_data['CODE'].str.upper().str.strip()
coverage_data['ANTIGEN'] = coverage_data['ANTIGEN'].str.upper().str.strip()
coverage_data['COVERAGE_CATEGORY'] = coverage_data['COVERAGE_CATEGORY'].str.upper().str.strip()
print(f"  ✓ Standardized CODE, ANTIGEN, COVERAGE_CATEGORY")

print("\n2.5: Converting YEAR to integer...")
coverage_data['YEAR'] = coverage_data['YEAR'].astype(int)
print(f"  ✓ Converted YEAR from float to int")

print("\n2.6: Adding data quality flags...")
coverage_data['has_admin_data'] = (
    coverage_data['TARGET_NUMBER'].notna() & 
    coverage_data['DOSES'].notna()
)
coverage_data['data_completeness'] = np.where(
    coverage_data['has_admin_data'] & coverage_data['COVERAGE'].notna(), 'Complete',
    np.where(coverage_data['COVERAGE'].notna(), 'Coverage_Only',
    np.where(coverage_data['has_admin_data'], 'Admin_Only', 'Missing'))
)
print(f"  ✓ Added has_admin_data and data_completeness flags")


print("\n" + "="*80)
print("STEP 3: CLEANING INCIDENCE RATE DATA")
print("="*80)

print("\n3.1: Removing rows with NULL in key columns...")
before = len(incidence_rate)
incidence_rate = incidence_rate.dropna(subset=['CODE', 'YEAR', 'DISEASE'])
after = len(incidence_rate)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

print("\n3.2: Standardizing text columns...")
incidence_rate['CODE'] = incidence_rate['CODE'].str.upper().str.strip()
incidence_rate['DISEASE'] = incidence_rate['DISEASE'].str.upper().str.strip()
print(f"  ✓ Standardized CODE, DISEASE")

print("\n3.3: Converting YEAR to integer...")
incidence_rate['YEAR'] = incidence_rate['YEAR'].astype(int)
print(f"  ✓ Converted YEAR")

print("\n3.4: Checking for negative incidence rates...")
negative_count = (incidence_rate['INCIDENCE_RATE'] < 0).sum()
if negative_count > 0:
    incidence_rate.loc[incidence_rate['INCIDENCE_RATE'] < 0, 'INCIDENCE_RATE'] = np.nan
    print(f"  Fixed {negative_count} negative values")
else:
    print(f"  ✓ No negative values found")


print("\n" + "="*80)
print("STEP 4: CLEANING REPORTED CASES DATA")
print("="*80)

print("\n4.1: Removing rows with NULL in key columns...")
before = len(reported_cases)
reported_cases = reported_cases.dropna(subset=['CODE', 'YEAR', 'DISEASE'])
after = len(reported_cases)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

print("\n4.2: Standardizing text columns...")
reported_cases['CODE'] = reported_cases['CODE'].str.upper().str.strip()
reported_cases['DISEASE'] = reported_cases['DISEASE'].str.upper().str.strip()
print(f"  ✓ Standardized CODE, DISEASE")

print("\n4.3: Converting YEAR to integer...")
reported_cases['YEAR'] = reported_cases['YEAR'].astype(int)
print(f"  ✓ Converted YEAR")

print("\n4.4: Checking for negative case counts...")
negative_count = (reported_cases['CASES'] < 0).sum()
if negative_count > 0:
    reported_cases.loc[reported_cases['CASES'] < 0, 'CASES'] = np.nan
    print(f"  Fixed {negative_count} negative values")
else:
    print(f"  ✓ No negative values found")

print("\n" + "="*80)
print("STEP 5: CLEANING VACCINE INTRODUCTION DATA")
print("="*80)

print("\n5.1: Removing rows with NULL in key columns...")
before = len(vaccine_intro)
vaccine_intro = vaccine_intro.dropna(subset=['ISO_3_CODE', 'YEAR', 'DESCRIPTION'])
after = len(vaccine_intro)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

print("\n5.2: Standardizing text columns...")
vaccine_intro['ISO_3_CODE'] = vaccine_intro['ISO_3_CODE'].str.upper().str.strip()
vaccine_intro['DESCRIPTION'] = vaccine_intro['DESCRIPTION'].str.upper().str.strip()
print(f"  ✓ Standardized ISO_3_CODE, DESCRIPTION")

print("\n5.3: Converting YEAR to integer...")
vaccine_intro['YEAR'] = vaccine_intro['YEAR'].astype(int)
print(f"  ✓ Converted YEAR")


print("\n" + "="*80)
print("STEP 6: CLEANING VACCINE SCHEDULE DATA")
print("="*80)

print("\n6.1: Removing rows with NULL in key columns...")
before = len(vaccine_schedule)
vaccine_schedule = vaccine_schedule.dropna(subset=['ISO_3_CODE', 'YEAR', 'VACCINECODE', 'SCHEDULEROUNDS'])
after = len(vaccine_schedule)
print(f"  Removed: {before - after:,} rows")
print(f"  Remaining: {after:,} rows")

print("\n6.2: Standardizing text columns...")
vaccine_schedule['ISO_3_CODE'] = vaccine_schedule['ISO_3_CODE'].str.upper().str.strip()
vaccine_schedule['VACCINECODE'] = vaccine_schedule['VACCINECODE'].str.upper().str.strip()
print(f"  ✓ Standardized ISO_3_CODE, VACCINECODE")

print("\n6.3: Converting YEAR to integer...")
vaccine_schedule['YEAR'] = vaccine_schedule['YEAR'].astype(int)
print(f"  ✓ Converted YEAR")

print("\n" + "="*80)
print("STEP 7: VERIFYING PRIMARY KEYS")
print("="*80)

def verify_pk(df, cols, name):
    total = len(df)
    unique = df[cols].drop_duplicates().shape[0]
    is_unique = total == unique
    status = "✓ UNIQUE" if is_unique else "✗ NOT UNIQUE"
    print(f"{name:<30} {unique:>10,} / {total:>10,}  {status}")
    return is_unique

print("\nPrimary Key Verification:")
print("-"*80)
pk_coverage = verify_pk(coverage_data, ['CODE', 'YEAR', 'ANTIGEN', 'COVERAGE_CATEGORY'], 'Coverage Data')
pk_incidence = verify_pk(incidence_rate, ['CODE', 'YEAR', 'DISEASE'], 'Incidence Rate')
pk_cases = verify_pk(reported_cases, ['CODE', 'YEAR', 'DISEASE'], 'Reported Cases')
pk_intro = verify_pk(vaccine_intro, ['ISO_3_CODE', 'YEAR', 'DESCRIPTION'], 'Vaccine Introduction')
pk_schedule = verify_pk(vaccine_schedule, ['ISO_3_CODE', 'YEAR', 'VACCINECODE', 'SCHEDULEROUNDS', 'TARGETPOP'], 'Vaccine Schedule')

print("\n" + "="*80)
print("STEP 8: CLEANING SUMMARY REPORT")
print("="*80)

summary_data = {
    'Table': ['Coverage Data', 'Incidence Rate', 'Reported Cases', 'Vaccine Introduction', 'Vaccine Schedule'],
    'Original_Rows': [original_counts['coverage'], original_counts['incidence'], original_counts['cases'], original_counts['intro'], original_counts['schedule']],
    'Cleaned_Rows': [len(coverage_data), len(incidence_rate), len(reported_cases), len(vaccine_intro), len(vaccine_schedule)],
    'Rows_Removed': [
        original_counts['coverage'] - len(coverage_data),
        original_counts['incidence'] - len(incidence_rate),
        original_counts['cases'] - len(reported_cases),
        original_counts['intro'] - len(vaccine_intro),
        original_counts['schedule'] - len(vaccine_schedule)
    ]
}

summary_df = pd.DataFrame(summary_data)
summary_df['Retention_%'] = (summary_df['Cleaned_Rows'] / summary_df['Original_Rows'] * 100).round(2)

print("\n" + summary_df.to_string(index=False))

print("\n\nData Quality Improvements:")
print("-"*80)
print(f"✓ Fixed {negative_mask.sum():,} negative DOSES values")
print(f"✓ Fixed {severe_count:,} severe coverage errors (>150%)")
print(f"✓ Capped {moderate_count:,} moderate overcounting (100-150%)")
print(f"✓ Standardized all country codes and text fields")
print(f"✓ Converted YEAR columns from float to integer")
print(f"✓ Added data quality flags to coverage data")

print("\n" + "="*80)
print("STEP 9: EXPORTING CLEANED DATA")
print("="*80)

output_folder = 'cleaned_data/'
import os
os.makedirs(output_folder, exist_ok=True)

coverage_data.to_csv(f'{output_folder}coverage_data_cleaned.csv', index=False)
incidence_rate.to_csv(f'{output_folder}incidence_rate_cleaned.csv', index=False)
reported_cases.to_csv(f'{output_folder}reported_cases_cleaned.csv', index=False)
vaccine_intro.to_csv(f'{output_folder}vaccine_introduction_cleaned.csv', index=False)
vaccine_schedule.to_csv(f'{output_folder}vaccine_schedule_cleaned.csv', index=False)

print(f"\n✓ All cleaned files exported to '{output_folder}' folder:")
print(f"  • coverage_data_cleaned.csv")
print(f"  • incidence_rate_cleaned.csv")
print(f"  • reported_cases_cleaned.csv")
print(f"  • vaccine_introduction_cleaned.csv")
print(f"  • vaccine_schedule_cleaned.csv")

summary_df.to_csv(f'{output_folder}cleaning_summary_report.csv', index=False)
print(f"  • cleaning_summary_report.csv")

print("\n" + "="*80)
print("CLEANING COMPLETE!")
print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

print("\n" + "="*80)
print("NEXT STEPS")
print("="*80)


VACCINATION DATA CLEANING SCRIPT
Started: 2026-10-15 21:39:48

STEP 1: Loading data files...
✓ Coverage Data: 399,859 rows loaded
✓ Incidence Rate: 84,946 rows loaded
✓ Reported Cases: 84,870 rows loaded
✓ Vaccine Introduction: 138,321 rows loaded
✓ Vaccine Schedule: 8,053 rows loaded

STEP 2: CLEANING COVERAGE DATA

2.1: Removing rows with NULL in key columns...
  Removed: 1 rows
  Remaining: 399,858 rows

2.2: Fixing negative DOSES values...
  Found 8 negative values
  Range: -222,288,203 to -3,333
  ✓ Converted to NULL

2.3: Fixing impossible COVERAGE percentages...
  Severe errors (>150%): 95 rows
  Moderate (100-150%): 5,002 rows
  ✓ Set >150% to NULL
  ✓ Capped 100-150% values at 100%

2.4: Standardizing text columns...
  ✓ Standardized CODE, ANTIGEN, COVERAGE_CATEGORY

2.5: Converting YEAR to integer...
  ✓ Converted YEAR from float to int

2.6: Adding data quality flags...
  ✓ Added has_admin_data and data_completeness flags

STEP 3: CLEANING INCIDENCE RATE DATA

3.1: Removing 